<a href="https://colab.research.google.com/github/Kilgor66/fooocus-colab/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Automated Fooocus Colab Launcher (Persistent Drive + Custom Model + Patch Fix)
# ==============================================================================

import os
from google.colab import drive

# 1. Mount Google Drive for persistent model and output storage
drive.mount('/content/drive')

# 2. Set working directory to Google Drive
%cd /content/drive/MyDrive/

# 3. Clone Fooocus repository if not already present
if not os.path.exists('/content/drive/MyDrive/Fooocus'):
    !git clone https://github.com/lllyasviel/Fooocus.git

%cd /content/drive/MyDrive/Fooocus

# 4. Apply fix for PyTorch no_init_weights bug (patch_clip.py)
!git checkout modules/patch_clip.py > /dev/null 2>&1
!sed -i 's/with modeling_utils.no_init_weights():/if True:/g' modules/patch_clip.py

# 5. Download custom Civitai model (talmendoxlSDXL_v11Beta) if it does not exist
model_path = "/content/drive/MyDrive/Fooocus/models/checkpoints/talmendoxlSDXL_v11Beta.safetensors"
if not os.path.exists(model_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://civitai.com/api/download/models/131960?fileId=95854" -d /content/drive/MyDrive/Fooocus/models/checkpoints -o talmendoxlSDXL_v11Beta.safetensors

# 6. Set custom default SDXL checkpoint preset
!git checkout presets/default.json > /dev/null 2>&1
!sed -i 's/juggernautXL_v8Rundiffusion.safetensors/talmendoxlSDXL_v11Beta.safetensors/g' presets/default.json

# 7. Launch application with public URL and High-VRAM mode
!source /content/env/bin/activate && unset MPLBACKEND && python entry_with_update.py --share --always-high-vram